In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, lfilter
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# SHANKS METHOD FOR IIR FILTER DESIGN
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.sh-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.sh-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.sh-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.sh-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.sh-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.sh-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.sh-col{
    flex:1;
    min-width:0;
}

.sh-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.sh-code{
    font-family:Consolas,monospace;
    font-size:13px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="sh-root">

<div class="sh-header">
Shanks Method for IIR Filter Design
</div>

<div class="sh-doc">

The Shanks method extends the Prony approximation by applying least-squares
optimization to both parts of the rational model.

The approximating filter is written as the cascade

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H(z) = H₁(z) H₂(z),
</b>
</div>

where

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H₁(z) = 1 /
(1 + α₁z<sup>-1</sup> + ... + α<sub>N</sub>z<sup>-N</sup>)
</b>
</div>

contains only poles, while

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H₂(z) =
β₀ + β₁z<sup>-1</sup> + ... + β<sub>M</sub>z<sup>-M</sup>
</b>
</div>

contains only zeros.

The denominator coefficients αₖ are first obtained by the same least-squares
procedure used in the Prony method. The impulse response v[n] of H₁(z) is then
used as the input of H₂(z), giving

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
ĥ<sub>d</sub>[n] =
Σ βₖv[n-k].
</b>
</div>

Unlike Prony, the numerator coefficients βₖ are also obtained by minimizing
the total squared approximation error over the available samples.

The example below uses the same ideal low-pass target with
<b>M = N = 5</b>.

</div>

</div>
"""))

# ============================================================
# DESIRED IMPULSE-RESPONSE SAMPLES
# ============================================================

hd = np.array([0.063661,0.000000,-0.106103,0.000000,0.318309,0.500000,0.318309,0.000000,-0.106103,0.000000,0.063661])

M = 5
N = 5

# ============================================================
# DENOMINATOR COEFFICIENTS FROM THE PRONY LS STAGE
# ============================================================

Xi_alpha = np.array([
    [0.362578,0.318309,0.067547,-0.106103,-0.067547],
    [0.318309,0.463899,0.318309,0.033774,-0.106103],
    [0.067547,0.318309,0.452641,0.318309,0.067547],
    [-0.106103,0.033774,0.318309,0.463899,0.318309],
    [-0.067547,-0.106103,0.067547,0.318309,0.362578]
])

gamma_alpha = np.array([-0.159155,-0.060792,0.053052,0.047283,-0.031830])

alpha = np.linalg.solve(Xi_alpha,gamma_alpha)

a = np.concatenate(([1.0],alpha))

# ============================================================
# ALL-POLE SUBSYSTEM H1(z)
# ============================================================

L = 11

impulse = np.zeros(L)

impulse[0] = 1.0

v = lfilter([1.0],a,impulse)

# ============================================================
# CONVOLUTION MATRIX FOR NUMERATOR LEAST-SQUARES DESIGN
# ============================================================

V = np.zeros((L,M+1))

for n in range(L):

    for k in range(M+1):

        if n-k >= 0:

            V[n,k] = v[n-k]

# ============================================================
# NORMAL EQUATIONS FOR BETA
# ============================================================

Xi_beta = V.T@V

gamma_beta = V.T@hd

beta = np.linalg.solve(Xi_beta,gamma_beta)

b = beta.copy()

# ============================================================
# APPROXIMATED IMPULSE RESPONSE OVER AVAILABLE SAMPLES
# ============================================================

h_hat_short = V@beta

error_short = hd-h_hat_short

E_short = np.sum(error_short**2)

# ============================================================
# POLES, ZEROS, AND STABILITY
# ============================================================

poles = np.roots(a)

zeros = np.roots(b)

max_pole_radius = np.max(np.abs(poles))

stable = max_pole_radius < 1.0

# ============================================================
# LONGER IMPULSE RESPONSE
# ============================================================

L_long = 40

impulse_long = np.zeros(L_long)

impulse_long[0] = 1.0

h_shanks = lfilter(b,a,impulse_long)

# ============================================================
# DESIRED IDEAL LOW-PASS IMPULSE RESPONSE
# ============================================================

n_long = np.arange(L_long)

hd_long = np.zeros(L_long)

for i,n in enumerate(n_long):

    if n == 5:

        hd_long[i] = 0.5

    else:

        hd_long[i] = np.sin((n-5)*np.pi/2)/(np.pi*(n-5))

error_long = hd_long-h_shanks

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H = freqz(b,a,worN=32768)

omega_norm = omega/np.pi

mag = np.abs(H)

ideal_mag = np.where(omega <= np.pi/2,1.0,0.0)

# ============================================================
# DISPLAY — NUMERICAL EXAMPLE
# ============================================================

display(HTML(f"""
<div class="sh-root">

<div class="sh-box sh-note">

<div class="sh-title">Numerical example — Ideal low-pass approximation</div>

The first eleven desired impulse-response samples are

<div class="sh-code" style="margin-top:5px;text-align:center;">
[0.063661, 0, -0.106103, 0, 0.318309, 0.500000,
0.318309, 0, -0.106103, 0, 0.063661]
</div>

The Shanks approximation uses

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>M = 5 zeros, &nbsp;&nbsp; N = 5 poles.</b>
</div>

</div>

<div class="sh-box">

<div class="sh-title">Calculated Shanks model</div>

<div class="sh-cols">

<div class="sh-col">

<b>Denominator coefficients</b><br>

α₀ = {a[0]:.6f}<br>
α₁ = {a[1]:.6f}<br>
α₂ = {a[2]:.6f}<br>
α₃ = {a[3]:.6f}<br>
α₄ = {a[4]:.6f}<br>
α₅ = {a[5]:.6f}

</div>

<div class="sh-col">

<b>Numerator coefficients</b><br>

β₀ = {b[0]:.6f}<br>
β₁ = {b[1]:.6f}<br>
β₂ = {b[2]:.6f}<br>
β₃ = {b[3]:.6f}<br>
β₄ = {b[4]:.6f}<br>
β₅ = {b[5]:.6f}

</div>

<div class="sh-col">

<b>Model diagnostics</b><br>

Maximum pole radius:<br>

<b>{max_pole_radius:.6f}</b><br><br>

Stable:
<b>{"YES" if stable else "NO"}</b><br><br>

Least-squares error:<br>

<b>E = {E_short:.3e}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# TRANSFER FUNCTION
# ============================================================

display(HTML(f"""
<div class="sh-root">

<div class="sh-box sh-note">

<div class="sh-title">Resulting transfer function</div>

<div style="font-size:14px;line-height:1.65;">

H(z) =
<b>
({b[0]:.6f}
{b[1]:+.6f}z<sup>-1</sup>
{b[2]:+.6f}z<sup>-2</sup>
{b[3]:+.6f}z<sup>-3</sup>
{b[4]:+.6f}z<sup>-4</sup>
{b[5]:+.6f}z<sup>-5</sup>) /
</b>

<br>

<div style="padding-left:48px;">

<b>
(1
{a[1]:+.6f}z<sup>-1</sup>
{a[2]:+.6f}z<sup>-2</sup>
{a[3]:+.6f}z<sup>-3</sup>
{a[4]:+.6f}z<sup>-4</sup>
{a[5]:+.6f}z<sup>-5</sup>)
</b>

</div>

</div>

<div style="margin-top:6px;">

The denominator is obtained from the Prony least-squares stage, while the
numerator coefficients are now also determined by least-squares minimization
over all available samples.

</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. ALL-POLE SUBSYSTEM IMPULSE RESPONSE
# ============================================================

n_short = np.arange(L)

marker_v,stem_v,base_v = ax1.stem(n_short,v,linefmt='r-',markerfmt='ro',basefmt=' ')

plt.setp(stem_v,linewidth=1.1)

marker_v.set_markersize(4.5)

ax1.axhline(0,color='black',linewidth=0.8)

ax1.set_xlim(-0.5,L-0.5)

ax1.set_ylim(-1.2,3.2)

ax1.set_title(r'Impulse Response $v[n]$ of $H_1(z)$')

ax1.set_xlabel('Sample index n')

ax1.set_ylabel('Amplitude')

ax1.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# 2. FIRST 11 SAMPLES — DESIRED VS SHANKS
# ============================================================

marker_hd,stem_hd,base_hd = ax2.stem(n_short,hd,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_h,stem_h,base_h = ax2.stem(n_short,h_hat_short,linefmt='r--',markerfmt='ro',basefmt=' ')

plt.setp(stem_hd,linewidth=1.1)

plt.setp(stem_h,linewidth=1.0)

marker_hd.set_markersize(4.5)

marker_h.set_markersize(3.5)

marker_hd.set_label(r'Desired $h_d[n]$')

marker_h.set_label(r'Shanks $\hat{h}_d[n]$')

ax2.axhline(0,color='black',linewidth=0.8)

ax2.set_xlim(-0.5,L-0.5)

ax2.set_ylim(-0.2,0.58)

ax2.set_title('Least-Squares Fit of the First 11 Samples')

ax2.set_xlabel('Sample index n')

ax2.set_ylabel('Amplitude')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 3. LONGER IMPULSE-RESPONSE COMPARISON
# ============================================================

ax3.plot(n_long,hd_long,color='black',linewidth=1.2,label=r'Desired $h_d[n]$')

ax3.plot(n_long,h_shanks,color='red',linewidth=1.3,label='Shanks')

ax3.axhline(0,color='black',linewidth=0.8)

ax3.set_xlim(0,L_long-1)

ax3.set_ylim(-0.8,0.8)

ax3.set_title('Desired vs Shanks Impulse Response')

ax3.set_xlabel('Sample index n')

ax3.set_ylabel('Amplitude')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 4. MAGNITUDE RESPONSE
# ============================================================

ax4.plot(omega_norm,ideal_mag,color='black',linewidth=1.2,label='Ideal low-pass')

ax4.plot(omega_norm,mag,color='red',linewidth=1.4,label='Shanks approximation')

ax4.axvline(0.5,linestyle='--',linewidth=1.0,label=r'$\omega_c=\pi/2$')

ax4.set_xlim(0,1)

ax4.set_ylim(0,3.0)

ax4.set_title('Ideal vs Shanks Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel(r'$|H(e^{j\omega})|$')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)